In [6]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split

In [7]:
df = pd.read_csv('dataset/spam_email_dataset.csv')
# BERT need long text, so we combine subject and email text into one column
df['combined_text'] = "Subject: " + df['subject'].astype(str) + " \nBody: " + df['email_text'].astype(str)
# Use only the combined text and label for model training
df_model = df[['combined_text', 'label']].rename(columns={'combined_text': 'text'})

In [8]:
# Train Eval Split
train_df, eval_df = train_test_split(df_model, test_size=0.2, random_state=42, stratify=df_model['label'])
# Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

In [9]:
# Load Tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_eval = eval_dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 2000/2000 [00:00<00:00, 5623.93 examples/s]


In [10]:
# Create Model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# Train Model
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,              # Epochs
    per_device_train_batch_size=8,   # Batch Size
    eval_strategy="epoch",     # Eval every epoch
    logging_dir='./logs',            
)

# Trainer API (No PyTorch Loop Needed)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
)

# Train
trainer.train()

# Save Model and Tokenizer
model.save_pretrained("./spam_detector_model")
tokenizer.save_pretrained("./spam_detector_model")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 679.03it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
C:\Users\Acer\AppData\Roaming\Pyt

Epoch,Training Loss,Validation Loss
1,0.000014,0.000004
2,0.000002,0.000001
3,0.000001,0.000000
4,0.000001,0.000000
5,0.000000,0.000000


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.11s/it]
C:\Users\Acer\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.16it/s]
C:\Users\Acer\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.50s/it]
C:\Users\Acer\AppData\Roaming\Python\Python313\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.63it/s]

('./spam_detector_model\\tokenizer_config.json',
 './spam_detector_model\\tokenizer.json')